<a href="https://colab.research.google.com/github/JPAmewu/My_Capstone_1_Imperial/blob/main/Week_6_Capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Week 6 Capstone Queries

# This notebook prepares the Week 6 dataset by appending the Week 5 query inputs and returned outputs, then uses Bayesian Optimisation to generate the next Week 6 query points.

In [ ]:
# =========================
# 1. Import libraries
# =========================

import os
import numpy as np
import pandas as pd

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, WhiteKernel, ConstantKernel
from sklearn.preprocessing import StandardScaler
from scipy.stats import norm

print("Libraries imported successfully.")

Libraries imported successfully.


In [ ]:
# =========================
# 2. Check files and folders
# =========================

print("Current files in /content:")
print(os.listdir("/content"))

print("\nFiles inside week_5_data:")
print(os.listdir("/content/week_5_data"))

print("\nFiles inside week_6_data:")
print(os.listdir("/content/week_6_data"))

Current files in /content:
['.config', 'Week_5_inputs.txt', '.ipynb_checkpoints', 'week_5_data', 'Week_5_outputs.txt', 'drive', 'week_6_data', 'sample_data']

Files inside week_5_data:
['function_8_inputs.npy', 'function_7_inputs.npy', 'function_1_outputs.npy', 'function_6_inputs.npy', 'function_7_outputs.npy', 'function_5_inputs.npy', 'function_4_inputs.npy', 'function_5_outputs.npy', 'function_8_outputs.npy', 'function_3_inputs.npy', 'function_6_outputs.npy', 'function_3_outputs.npy', 'function_2_outputs.npy', 'function_1_inputs.npy', 'function_2_inputs.npy', 'function_4_outputs.npy']

Files inside week_6_data:
['function_8_inputs.npy', 'function_7_inputs.npy', 'function_1_outputs.npy', 'function_6_inputs.npy', 'function_7_outputs.npy', 'function_5_inputs.npy', 'function_4_inputs.npy', 'function_5_outputs.npy', 'function_8_outputs.npy', 'function_3_inputs.npy', 'function_6_outputs.npy', 'function_3_outputs.npy', 'function_2_outputs.npy', 'function_1_inputs.npy', 'function_2_inputs.np

In [ ]:
# =========================
# 3. Check Week 6 data shapes
# =========================

import os
import numpy as np
import pandas as pd

week6_folder = "/content/week_6_data"

summary = []

for f in range(1, 9):
    x_path = f"{week6_folder}/function_{f}_inputs.npy"
    y_path = f"{week6_folder}/function_{f}_outputs.npy"

    if os.path.exists(x_path) and os.path.exists(y_path):
        X = np.load(x_path)
        y = np.load(y_path)

        summary.append({
            "Function": f"Function {f}",
            "Input shape": X.shape,
            "Output shape": y.shape,
            "Best output so far": np.max(y),
            "Best point index": np.argmax(y)
        })
    else:
        summary.append({
            "Function": f"Function {f}",
            "Input shape": "Missing",
            "Output shape": "Missing",
            "Best output so far": "Missing",
            "Best point index": "Missing"
        })

summary_df = pd.DataFrame(summary)
summary_df

,Function,Input shape,Output shape,Best output so far,Best point index
0,Function 1,"(15, 2)","(15,)",64.000000,10
1,Function 2,"(15, 2)","(15,)",64.000000,13
2,Function 3,"(20, 3)","(20,)",64.000000,15
3,Function 4,"(35, 4)","(35,)",64.000000,33
4,Function 5,"(25, 4)","(25,)",1088.859618,15
5,Function 6,"(25, 5)","(25,)",64.000000,23
6,Function 7,"(35, 6)","(35,)",64.000000,30
7,Function 8,"(45, 8)","(45,)",64.000000,43


In [ ]:
# =========================
# 5. Check for NaN values
# =========================

import os
import numpy as np
import pandas as pd

week6_folder = "/content/week_6_data"

nan_report = []

for f in range(1, 9):
    X = np.load(f"{week6_folder}/function_{f}_inputs.npy")
    y = np.load(f"{week6_folder}/function_{f}_outputs.npy").ravel()

    nan_report.append({
        "Function": f"Function {f}",
        "Input shape": X.shape,
        "Output shape": y.shape,
        "NaN in inputs": np.isnan(X).sum(),
        "NaN in outputs": np.isnan(y).sum(),
        "Infinite in inputs": np.isinf(X).sum(),
        "Infinite in outputs": np.isinf(y).sum()
    })

nan_report_df = pd.DataFrame(nan_report)
nan_report_df

,Function,Input shape,Output shape,NaN in inputs,NaN in outputs,Infinite in inputs,Infinite in outputs
0,Function 1,"(15, 2)","(15,)",0,0,0,0
1,Function 2,"(15, 2)","(15,)",0,0,0,0
2,Function 3,"(20, 3)","(20,)",0,0,0,0
3,Function 4,"(35, 4)","(35,)",0,0,0,0
4,Function 5,"(25, 4)","(25,)",0,0,0,0
5,Function 6,"(25, 5)","(25,)",0,0,0,0
6,Function 7,"(35, 6)","(35,)",0,0,0,0
7,Function 8,"(45, 8)","(45,)",1,0,0,0


In [ ]:
# =========================
# 6. Generate Week 6 query points safely
# =========================

import os
import numpy as np
import pandas as pd
import warnings

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, RBF, WhiteKernel, ConstantKernel
from sklearn.preprocessing import StandardScaler
from scipy.stats import norm

warnings.filterwarnings("ignore")

week6_folder = "/content/week_6_data"

# -------------------------
# Acquisition functions
# -------------------------

def expected_improvement(mu, sigma, best_y, xi=0.01):
    sigma = sigma.reshape(-1)
    mu = mu.reshape(-1)

    improvement = mu - best_y - xi
    Z = improvement / (sigma + 1e-9)

    ei = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)
    ei[sigma == 0.0] = 0.0

    return ei


def upper_confidence_bound(mu, sigma, kappa=2.5):
    return mu + kappa * sigma


# -------------------------
# Candidate generator
# -------------------------

def generate_candidates(best_x, dim, n_random=8000, n_local=3000, local_scale=0.08):
    random_candidates = np.random.rand(n_random, dim)

    local_candidates = best_x + np.random.normal(
        loc=0.0,
        scale=local_scale,
        size=(n_local, dim)
    )

    local_candidates = np.clip(local_candidates, 0, 1)

    candidates = np.vstack([random_candidates, local_candidates])

    return candidates


# -------------------------
# Clean X and y
# -------------------------

def clean_xy(X, y):
    y = y.ravel()

    valid_rows = np.isfinite(X).all(axis=1) & np.isfinite(y)

    X_clean = X[valid_rows]
    y_clean = y[valid_rows]

    removed_rows = len(y) - len(y_clean)

    return X_clean, y_clean, removed_rows


# -------------------------
# Main BO function
# -------------------------

def propose_next_point(X, y, function_number):
    X, y, removed_rows = clean_xy(X, y)

    if len(y) < 2:
        raise ValueError(f"Function {function_number} does not have enough clean data after removing NaNs.")

    dim = X.shape[1]

    best_index = np.argmax(y)
    best_x = X[best_index]
    best_y = y[best_index]

    y_scaler = StandardScaler()
    y_scaled = y_scaler.fit_transform(y.reshape(-1, 1)).ravel()

    if function_number in [1, 2, 3, 4]:
        kernel = ConstantKernel(1.0) * Matern(length_scale=np.ones(dim), nu=2.5) + WhiteKernel(noise_level=1e-5)
        acquisition_type = "UCB"
        kappa = 2.5

    elif function_number == 5:
        kernel = ConstantKernel(1.0) * RBF(length_scale=np.ones(dim)) + WhiteKernel(noise_level=1e-5)
        acquisition_type = "EI"
        kappa = None

    elif function_number in [6, 7]:
        kernel = ConstantKernel(1.0) * Matern(length_scale=np.ones(dim), nu=1.5) + WhiteKernel(noise_level=1e-5)
        acquisition_type = "UCB"
        kappa = 2.0

    else:
        kernel = ConstantKernel(1.0) * Matern(length_scale=np.ones(dim), nu=1.5) + WhiteKernel(noise_level=1e-5)
        acquisition_type = "EI"
        kappa = None

    gp = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=10,
        random_state=42
    )

    gp.fit(X, y_scaled)

    candidates = generate_candidates(best_x, dim)

    mu_scaled, sigma_scaled = gp.predict(candidates, return_std=True)

    best_y_scaled = y_scaled[best_index]

    if acquisition_type == "UCB":
        acquisition_values = upper_confidence_bound(mu_scaled, sigma_scaled, kappa=kappa)
    else:
        acquisition_values = expected_improvement(mu_scaled, sigma_scaled, best_y_scaled, xi=0.01)

    selected_index = np.argmax(acquisition_values)
    next_x = candidates[selected_index]

    return next_x, best_x, best_y, acquisition_type, removed_rows


# -------------------------
# Generate one Week 6 query per function
# -------------------------

week6_queries = {}
summary_rows = []

np.random.seed(42)

for f in range(1, 9):
    X = np.load(f"{week6_folder}/function_{f}_inputs.npy")
    y = np.load(f"{week6_folder}/function_{f}_outputs.npy").ravel()

    next_x, best_x, best_y, acquisition_type, removed_rows = propose_next_point(X, y, f)

    week6_queries[f"Function_{f}"] = next_x

    summary_rows.append({
        "Function": f"Function {f}",
        "Dimension": X.shape[1],
        "Current best output": best_y,
        "Acquisition used": acquisition_type,
        "Rows removed": removed_rows,
        "Week 6 query point": np.round(next_x, 6)
    })

week6_query_summary = pd.DataFrame(summary_rows)
week6_query_summary

,Function,Dimension,Current best output,Acquisition used,Rows removed,Week 6 query point
0,Function 1,2,64.000000,UCB,0,"[0.222521, 0.999673]"
1,Function 2,2,64.000000,UCB,0,"[0.473151, 0.950706]"
2,Function 3,3,64.000000,UCB,0,"[0.418776, 0.420613, 0.695421]"
3,Function 4,4,64.000000,UCB,0,"[0.753422, 0.441826, 0.942545, 0.489846]"
4,Function 5,4,1088.859618,EI,0,"[0.654981, 0.602894, 0.898891, 0.589622]"
5,Function 6,5,64.000000,UCB,0,"[0.78177, 0.192076, 0.805757, 0.695733, 0.56594]"
6,Function 7,6,64.000000,UCB,0,"[0.042873, 0.466831, 0.379623, 0.20709, 0.3677..."
7,Function 8,8,9.598482,EI,1,"[0.115956, 0.821931, 0.587418, 0.022809, 0.440..."


In [ ]:
# =========================
# 7. Show Week 6 queries in a clear table
# =========================

# Convert each query point into separate columns: x1, x2, x3, ...
expanded_rows = []

for f in range(1, 9):
    query = week6_queries[f"Function_{f}"]

    row = {
        "Function": f"Function {f}",
        "Dimension": len(query)
    }

    for i, value in enumerate(query, start=1):
        row[f"x{i}"] = round(value, 6)

    expanded_rows.append(row)

week6_queries_clear = pd.DataFrame(expanded_rows)

# Show all columns clearly
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

week6_queries_clear

,Function,Dimension,x1,x2,x3,x4,x5,x6,x7,x8
0,Function 1,2,0.222521,0.999673,NaN,NaN,NaN,NaN,NaN,NaN
1,Function 2,2,0.473151,0.950706,NaN,NaN,NaN,NaN,NaN,NaN
2,Function 3,3,0.418776,0.420613,0.695421,NaN,NaN,NaN,NaN,NaN
3,Function 4,4,0.753422,0.441826,0.942545,0.489846,NaN,NaN,NaN,NaN
4,Function 5,4,0.654981,0.602894,0.898891,0.589622,NaN,NaN,NaN,NaN
5,Function 6,5,0.781770,0.192076,0.805757,0.695733,0.565940,NaN,NaN,NaN
6,Function 7,6,0.042873,0.466831,0.379623,0.207090,0.367754,0.515646,NaN,NaN
7,Function 8,8,0.115956,0.821931,0.587418,0.022809,0.440260,0.965550,0.368459,0.301774


In [ ]:
# =========================
# 8. Print Week 6 query points line by line
# =========================

np.set_printoptions(precision=6, suppress=True)

for f in range(1, 9):
    print(f"Function {f}:")
    print(week6_queries[f"Function_{f}"])
    print()

Function 1:
[0.222521 0.999673]

Function 2:
[0.473151 0.950706]

Function 3:
[0.418776 0.420613 0.695421]

Function 4:
[0.753422 0.441826 0.942545 0.489846]

Function 5:
[0.654981 0.602894 0.898891 0.589622]

Function 6:
[0.78177  0.192076 0.805757 0.695733 0.56594 ]

Function 7:
[0.042873 0.466831 0.379623 0.20709  0.367754 0.515646]

Function 8:
[0.115956 0.821931 0.587418 0.022809 0.44026  0.96555  0.368459 0.301774]

